# RailGuard — Model Evaluation

This notebook consolidates the evaluation results for all three deep
learning models in RailGuard: the CNN classifier, the Autoencoder anomaly
detector, and the YOLOv8 defect detector. Metrics below are taken directly
from the training and evaluation runs performed in Google Colab.

Run the cells top to bottom. No GPU or trained model files are required to
view the summary tables and charts — they use the recorded metrics
directly. The optional "Live inference" section at the end can be run if
you have the trained model files available locally.

## 1. Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams['figure.figsize'] = (9, 5)
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3

## 2. CNN Classifier (MobileNetV2 Transfer Learning)

**Task:** Binary classification — Defective vs. Non defective track images
**Dataset:** Kaggle Railway Track Fault Detection (299 train / 62 val / 22 test)
**Training strategy:** Frozen MobileNetV2 backbone + custom head, with
EarlyStopping (patience=5, monitor=val_accuracy) and ModelCheckpoint to
prevent overfitting after an initial overfit run.

In [ ]:
# Training history recorded from the second (corrected) training run
cnn_history = {
    "epoch": list(range(1, 9)),
    "train_accuracy": [0.6120, 0.7124, 0.7893, 0.7692, 0.8629, 0.8027, 0.8294, None],
    "val_accuracy":   [0.7419, 0.8065, 0.7742, 0.8065, 0.7903, 0.7419, 0.7258, None],
}
df_cnn = pd.DataFrame(cnn_history)
df_cnn

In [ ]:
plt.plot(df_cnn["epoch"], df_cnn["train_accuracy"], marker="o", label="Train Accuracy")
plt.plot(df_cnn["epoch"], df_cnn["val_accuracy"], marker="o", label="Validation Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("CNN Training Curve (EarlyStopping run)")
plt.legend()
plt.show()

### Final Test Set Result

Evaluated on 22 held-out test images the model never saw during training or
validation.

In [ ]:
cnn_test_results = {
    "Metric": ["Test Accuracy", "Test Loss"],
    "Value": ["81.82%", "0.3955"]
}
pd.DataFrame(cnn_test_results)

**Interpretation:** Validation accuracy (80.65%) and test accuracy
(81.82%) are close to each other, indicating the model generalizes
consistently rather than performing well on one split by chance. This is a
genuinely reportable, non-overfit result.

## 3. Autoencoder (Anomaly Detection)

**Task:** Flag track images that deviate from "normal" (healthy) appearance
**Training data:** 150 healthy ("Non defective") images only
**Approach:** Reconstruction error (MSE) compared against a tuned threshold

In [ ]:
ae_training_summary = {
    "Epoch": [1, 5, 10, 15, 20, 25, 30],
    "Train Loss (MSE)": [0.0420, 0.0178, 0.0150, 0.0128, 0.0120, 0.0114, 0.0112],
    "Val Loss (MSE)":   [0.0461, 0.0203, 0.0174, 0.0151, 0.0142, 0.0137, 0.0134],
}
df_ae = pd.DataFrame(ae_training_summary)
df_ae

In [ ]:
plt.plot(df_ae["Epoch"], df_ae["Train Loss (MSE)"], marker="o", label="Train Loss")
plt.plot(df_ae["Epoch"], df_ae["Val Loss (MSE)"], marker="o", label="Validation Loss")
plt.xlabel("Epoch")
plt.ylabel("MSE Loss")
plt.title("Autoencoder Training Curve")
plt.legend()
plt.show()

### Threshold Tuning

The anomaly threshold was selected by comparing detection rate against
false-positive rate across several standard-deviation multipliers
(threshold = mean + k × std of reconstruction error on healthy validation
images).

In [ ]:
threshold_comparison = {
    "k (std multiplier)": [0.5, 1.0, 1.5, 2.0],
    "Threshold": [0.0156, 0.0178, 0.0200, 0.0222],
    "Healthy False Positives (/31)": [7, 3, 2, 2],
    "Defects Caught (/31)": [16, 13, 10, 8],
}
df_thresh = pd.DataFrame(threshold_comparison)
df_thresh["False Positive Rate"] = (df_thresh["Healthy False Positives (/31)"] / 31 * 100).round(1).astype(str) + "%"
df_thresh["Detection Rate"] = (df_thresh["Defects Caught (/31)"] / 31 * 100).round(1).astype(str) + "%"
df_thresh

In [ ]:
fig, ax = plt.subplots()
ax.plot(df_thresh["k (std multiplier)"], df_thresh["Healthy False Positives (/31)"] / 31 * 100,
        marker="o", label="False Positive Rate (%)")
ax.plot(df_thresh["k (std multiplier)"], df_thresh["Defects Caught (/31)"] / 31 * 100,
        marker="o", label="Detection Rate (%)")
ax.set_xlabel("k (standard deviation multiplier)")
ax.set_ylabel("Percentage")
ax.set_title("Autoencoder Threshold Tradeoff")
ax.legend()
plt.show()

**Selected threshold: k = 1.0** (threshold = 0.0178), balancing a
42% anomaly-detection rate against a 10% false-positive rate on healthy
images. This was chosen over more aggressive (k=0.5) or conservative
(k=1.5, 2.0) options as the best practical tradeoff — see project report
for full discussion.

## 4. YOLOv8 (Defect Detection)

**Task:** Localize and classify defects with bounding boxes
**Dataset:** Roboflow rail-track-defect v1 (36 train / 10 valid / 5 test
images, 5 classes)
**Note:** This dataset is very small for an object detection task —
results below should be read as a proof-of-concept, not a production-grade
detector. This limitation is discussed explicitly in the project report.

In [ ]:
yolo_per_class = {
    "Class": ["crack", "expand", "weak", "wear-tear"],
    "Images": [1, 3, 4, 4],
    "Instances": [1, 3, 5, 4],
    "Precision": [1.000, 0.916, 1.000, 1.000],
    "Recall": [0.000, 0.333, 0.000, 0.000],
    "mAP50": [0.0065, 0.344, 0.0222, 0.0108],
}
df_yolo = pd.DataFrame(yolo_per_class)
df_yolo

In [ ]:
df_yolo.set_index("Class")[["Precision", "Recall", "mAP50"]].plot(kind="bar")
plt.title("YOLOv8 Per-Class Performance")
plt.ylabel("Score")
plt.xticks(rotation=0)
plt.legend(loc="upper right")
plt.show()

**Overall metrics:** Precision = 0.979, Recall = 0.083, mAP50 = 0.096,
mAP50-95 = 0.062

**Interpretation:** High precision but very low recall means that when the
model does draw a bounding box, it is usually correct — but it misses the
large majority of actual defects. This is a direct consequence of the very
small number of training images per class (often fewer than 10), which is
insufficient for an object detector to learn robust localization. This is a
data-scale limitation, not a flaw in the training methodology, and is
consistent with the general finding in the literature that supervised
object detectors require substantially more labeled examples per class than
classification models to achieve reliable recall.

## 5. Cross-Model Summary

In [ ]:
summary = {
    "Model": ["CNN (MobileNetV2)", "Autoencoder", "YOLOv8n"],
    "Task": ["Binary classification", "Anomaly detection", "Object detection"],
    "Key Metric": ["Test Accuracy", "Detection Rate @ 10% FPR", "mAP50"],
    "Result": ["81.82%", "42%", "0.096 (9.6%)"],
    "Assessment": [
        "Reliable, presentable, evaluated on unseen test data",
        "Reasonable given small healthy-image training set",
        "Proof-of-concept only; limited by 36-image training set"
    ]
}
pd.DataFrame(summary)

## 6. (Optional) Live Inference on Trained Models

The cells below require the actual trained model files
(`best_cnn_model.keras`, `autoencoder.keras`, `ae_threshold.json`,
`yolo_defect_v2_best.pt`) to be present in a `models/` folder next to this
notebook. Skip this section if you only want the recorded metrics above.

In [ ]:
import os
import json

MODELS_DIR = "models"
required_files = [
    "best_cnn_model.keras", "autoencoder.keras",
    "ae_threshold.json", "yolo_defect_v2_best.pt"
]
missing = [f for f in required_files if not os.path.exists(os.path.join(MODELS_DIR, f))]

if missing:
    print("Skipping live inference -- missing files:", missing)
else:
    from tensorflow.keras.models import load_model
    from ultralytics import YOLO

    cnn_model = load_model(os.path.join(MODELS_DIR, "best_cnn_model.keras"))
    ae_model = load_model(os.path.join(MODELS_DIR, "autoencoder.keras"))
    yolo_model = YOLO(os.path.join(MODELS_DIR, "yolo_defect_v2_best.pt"))

    with open(os.path.join(MODELS_DIR, "ae_threshold.json")) as f:
        ae_threshold = json.load(f)["threshold"]

    print("All models loaded. AE threshold:", ae_threshold)
    print("Set TEST_IMAGE_PATH below and run the next cell to test on a real image.")

In [ ]:
# TEST_IMAGE_PATH = "path/to/your/test/image.jpg"
# Uncomment and run once TEST_IMAGE_PATH is set and models are loaded above.

# from tensorflow.keras.preprocessing import image as keras_image
#
# img_cnn = keras_image.load_img(TEST_IMAGE_PATH, target_size=(224, 224))
# arr_cnn = keras_image.img_to_array(img_cnn) / 255.0
# arr_cnn = np.expand_dims(arr_cnn, axis=0)
# pred = cnn_model.predict(arr_cnn, verbose=0)[0][0]
# print("CNN classification:", "Non defective" if pred > 0.5 else "Defective",
#       f"({pred:.3f})")
#
# yolo_results = yolo_model(TEST_IMAGE_PATH, conf=0.25, verbose=False)
# print("YOLO detections:", len(yolo_results[0].boxes))

---
*RailGuard — Railway Safety and Predictive Maintenance System*